# 00: Colab Setup & Environment Verification

This notebook prepares the environment for the research project:
**"An Evidence-Augmented Explainable Transformer Framework for Early Depression Risk Detection from Social Media Text"**

It is designed to be runnable safely multiple times without causing duplicate directory structures or errors.

## 1. Environment Detection and Repository Setup
Detect whether the code is running in Google Colab, clone the repository if necessary, and set the working directory to the repository root.

In [ ]:
import sys
import os
import subprocess

# Detect if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules
print(f"Running in Google Colab: {IN_COLAB}")

# Clone repository if not available
repo_url = "https://github.com/Rahad-Max/evidence-augmented-depression-transformer.git"
repo_name = "evidence-augmented-depression-transformer"

if IN_COLAB:
    if not os.path.exists(repo_name):
        print(f"Cloning repository: {repo_url}")
        subprocess.run(["git", "clone", repo_url])
    
    # Set working directory
    os.chdir(repo_name)
else:
    # If local, ensure we are in the root directory (assuming notebooks run from notebooks/)
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")

print(f"Working directory set to: {os.getcwd()}")

## 2. Install Dependencies
Install necessary Python packages from `requirements.txt`.

In [ ]:
if IN_COLAB:
    print("Installing dependencies from requirements.txt...")
    !pip install -q -r requirements.txt
    print("Dependencies installed.")
else:
    print("Assuming local environment has dependencies installed. Run `pip install -r requirements.txt` if not.")

## 3. Check Python and PyTorch Installation
Verify the Python version, PyTorch installation, and GPU/CUDA availability.

In [ ]:
import platform
import torch

python_version = platform.python_version()
cuda_available = torch.cuda.is_available()
gpu_name = "None"
cuda_version = "None"

if cuda_available:
    gpu_name = torch.cuda.get_device_name(0)
    cuda_version = torch.version.cuda

print(f"Python version: {python_version}")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"GPU Name: {gpu_name}")
    print(f"CUDA version: {cuda_version}")

## 4. Configure Python Path and Verify Modules
Ensure that the `src/` directory can be imported and verify that all necessary custom modules are accessible.

In [ ]:
# Configure Python path to include the root directory so `src` can be imported
if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

# Import and verify main modules
modules_loaded = True
try:
    from src import preprocessing, models, retrieval, evidence, explainability, evaluation
    print("All custom source modules imported successfully:")
    print(" - preprocessing, models, retrieval, evidence, explainability, evaluation")
except ImportError as e:
    print(f"Error importing modules: {e}")
    modules_loaded = False

## 5. Create Required Directories
Creates necessary directories for storing results, metrics, figures, predictions, and processed data without overwriting existing files.

In [ ]:
required_dirs = [
    "results/metrics",
    "results/figures",
    "results/predictions",
    "data/processed"
]

for d in required_dirs:
    os.makedirs(d, exist_ok=True)
    
print("Required directories verified/created successfully.")

## 6. Set Reproducible Random Seeds
Loads the configuration file (`configs/experiment.yaml`) and applies the predefined seed across numpy, Python's random module, and PyTorch.

In [ ]:
import yaml
import numpy as np
import random

config_path = "configs/experiment.yaml"
seed = 42 # Default fallback seed

if os.path.exists(config_path):
    with open(config_path, "r") as f:
        try:
            config = yaml.safe_load(f)
            seed = config.get("seed", seed)
        except Exception as e:
            print(f"Warning: Could not parse {config_path}. Using default seed ({seed}).")
else:
    print(f"Warning: Configuration file {config_path} not found. Using default seed ({seed}).")

# Apply seeds
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

print(f"Random seed successfully set to: {seed}")

## 7. Environment Verification Report
A concise final report of the configuration.

In [ ]:
print("="*50)
print("ENVIRONMENT VERIFICATION REPORT")
print("="*50)
print(f"Repository:       {os.getcwd()}")
print(f"Python:           {python_version}")
print(f"PyTorch:          {torch.__version__}")
print(f"GPU:              {gpu_name}")
print(f"CUDA:             {cuda_available} (Version: {cuda_version})")
print(f"Required modules: {'SUCCESS' if modules_loaded else 'FAILED'}")
print(f"Directory struct: READY")
print(f"Environment stat: {'READY FOR EXPERIMENTS' if modules_loaded else 'SETUP INCOMPLETE'}")
print("="*50)
